# 내 모델 RAG 재파인튜닝 (LoRA)

`jjminu/polyglot58-chatbot` 위에 **LoRA**로 RAG 행동을 학습시킨다:
1. **문서 근거로만 답하기**  2. **문서에 없으면 거부하기**

학습셋 51개 (근거답변 39 + 거부 12) — 내 docs 사실 기반.

> **런타임 유형 = GPU (A100 또는 L4 권장)** 로 설정 후 위에서부터 실행.

## 1. 설치

In [ ]:
# torchao 제거: 구버전(0.10)이 새 peft 와 충돌. 우리는 bitsandbytes 만 쓰므로 불필요
!pip -q uninstall -y torchao
!pip -q install -U transformers peft accelerate bitsandbytes datasets

## 2. GPU 확인

In [ ]:
import torch
print('CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Hugging Face 로그인 (학습 결과 업로드용)
쓰기 권한 토큰 필요: https://huggingface.co/settings/tokens (Write)

In [ ]:
from huggingface_hub import login
import getpass
login(getpass.getpass('HF Write Token (hf_...): '))

## 4. 학습 데이터 (내 문서 기반)

In [ ]:
train_rows = [
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "어댑터즈 월 구독료는 얼마인가요?",
  "answer": "월 구독료는 1만 9천 원입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "어댑터즈 구독료 얼마야?",
  "answer": "월 1만 9천 원입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "어댑터즈는 어떤 서비스야?",
  "answer": "현업 개발자가 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "어댑터즈 교재는 얼마나 자주 업데이트돼?",
  "answer": "매주 새로 업데이트됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "어댑터즈 코드 예제는 어디서 받아?",
  "answer": "깃허브 저장소로 함께 제공됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.",
  "question": "스타트업코드는 언제 설립됐어?",
  "answer": "2021년에 설립되었습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.",
  "question": "누적 수강생은 몇 명이야?",
  "answer": "1만 2천 명을 넘었습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.",
  "question": "스타트업코드는 어떤 회사야?",
  "answer": "개발자를 위한 교육과 교재를 제공하는 회사입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.",
  "question": "스타트업코드 주력 서비스가 뭐야?",
  "answer": "개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "부트캠프는 몇 주 과정이야?",
  "answer": "16주 과정입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "부트캠프 트랙은 어떤 게 있어?",
  "answer": "백엔드 트랙과 AI 트랙 두 가지가 있습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "백엔드 트랙은 뭘 배워?",
  "answer": "파이썬과 FastAPI를 배웁니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "AI 트랙은 뭘 다뤄?",
  "answer": "딥러닝과 LLM 애플리케이션 개발을 다룹니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "수료 조건이 뭐야?",
  "answer": "최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "부트캠프 수강료는 얼마야?",
  "answer": "350만 원이며 분할 납부가 가능합니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "수강료 분할 납부 돼?",
  "answer": "네, 분할 납부가 가능합니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "환불은 언제까지 신청하면 전액 환불돼?",
  "answer": "수강 시작 후 14일 이내에 신청하면 전액 환불됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "14일 지나면 환불 어떻게 돼?",
  "answer": "진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서 첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.",
  "question": "수료하면 뭘 받아?",
  "answer": "수료증이 발급됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서 첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.",
  "question": "취업 지원은 어떤 게 있어?",
  "answer": "이력서 첨삭과 모의 면접을 제공하고 협력 기업에 추천서를 보내줍니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "문의는 어디로 해?",
  "answer": "이메일(support@startupcode.kr) 또는 카카오톡 채널로 문의할 수 있습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "상담 가능 시간은?",
  "answer": "평일 오전 10시부터 오후 6시까지입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "주말에도 문의 답변 받을 수 있어?",
  "answer": "주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] AI 트랙은 딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다. 수료 후에는 이미지 분류 모델과 RAG 기반 챗봇을 직접 만들 수 있는 수준을 목표로 합니다.",
  "question": "AI 트랙 수료하면 뭘 만들 수 있어?",
  "answer": "이미지 분류 모델과 RAG 기반 챗봇을 만들 수 있는 수준을 목표로 합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] AI 트랙은 딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다. 수료 후에는 이미지 분류 모델과 RAG 기반 챗봇을 직접 만들 수 있는 수준을 목표로 합니다.",
  "question": "AI 트랙은 무엇부터 다뤄?",
  "answer": "딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.",
  "question": "1주차엔 뭘 배워?",
  "answer": "파이썬 복습과 넘파이를 다룹니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.",
  "question": "2주차부터 4주차까지는 뭘 배워?",
  "answer": "신경망과 CNN을 배우고 ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.",
  "question": "5주차엔 뭘 해?",
  "answer": "자연어 처리와 챗봇을 만들고 RAG 아키텍처를 적용합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.",
  "question": "전이학습은 어떤 모델로 실습해?",
  "answer": "ResNet과 VGG16 같은 사전학습 모델로 실습합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] RAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에 넣고 LLM이 답변을 생성합니다.",
  "question": "RAG 실습에서 임베딩은 뭘로 해?",
  "answer": "sentence-transformers로 임베딩합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] RAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에 넣고 LLM이 답변을 생성합니다.",
  "question": "임베딩한 건 어디에 저장해?",
  "answer": "ChromaDB에 저장합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] RAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에 넣고 LLM이 답변을 생성합니다.",
  "question": "관련 문서는 어떻게 검색해?",
  "answer": "코사인 유사도로 검색합니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 최종 프로젝트는 동작 여부, 코드 구조, 그리고 발표로 평가됩니다. 단순히 개념을 아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다.",
  "question": "최종 프로젝트는 뭘로 평가해?",
  "answer": "동작 여부, 코드 구조, 발표로 평가됩니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 최종 프로젝트는 동작 여부, 코드 구조, 그리고 발표로 평가됩니다. 단순히 개념을 아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다.",
  "question": "개념만 알면 높은 평가 받아?",
  "answer": "단순히 개념을 아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.\n\n[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "부트캠프 수강료 얼마야?",
  "answer": "350만 원이며 분할 납부가 가능합니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.\n\n[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "어댑터즈 구독료는?",
  "answer": "월 1만 9천 원입니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.\n\n[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "스타트업코드 설립 연도는?",
  "answer": "2021년입니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.\n\n[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "5주차에 뭐 배워?",
  "answer": "자연어 처리와 챗봇을 만들고 RAG 아키텍처를 적용합니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서 첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.\n\n[출처: ai_bootcamp_notes.md] RAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에 넣고 LLM이 답변을 생성합니다.",
  "question": "수료하면 수료증 주나요?",
  "answer": "네, 모든 과정을 수료하면 수료증이 발급됩니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "부트캠프 수강료 얼마야?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후 14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한 잔여 금액의 일부만 환불됩니다.",
  "question": "어댑터즈 구독료 얼마야?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가 있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션 개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.",
  "question": "환불 정책 알려줘",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "수강료 얼마야?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과 CNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다. 5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.",
  "question": "수료증 주나요?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는 개발 교재 플랫폼 어댑터즈와 온라인 코딩 부트캠프가 있습니다. 2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.",
  "question": "주말에 상담 돼?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] AI 트랙은 딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다. 수료 후에는 이미지 분류 모델과 RAG 기반 챗봇을 직접 만들 수 있는 수준을 목표로 합니다.",
  "question": "비트코인 사도 돼?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] RAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에 넣고 LLM이 답변을 생성합니다.",
  "question": "오늘 날씨 어때?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: ai_bootcamp_notes.md] 최종 프로젝트는 동작 여부, 코드 구조, 그리고 발표로 평가됩니다. 단순히 개념을 아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다.",
  "question": "어댑터즈 구독료는?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서 첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.",
  "question": "부트캠프 몇 주야?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr) 또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.",
  "question": "AI 트랙은 뭘 배워?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 },
 {
  "context": "[출처: startupcode_faq.md] 어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는 서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다. 교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.",
  "question": "강사 이름이 뭐야?",
  "answer": "제공된 문서에서 답을 찾을 수 없습니다."
 }
]
print('학습 예시:', len(train_rows), '개')
print(train_rows[0])

## 5. 베이스 모델 로드 (4-bit QLoRA) + LoRA 설정
내 5.8B 모델을 4비트로 올려 LoRA 어댑터만 학습한다 (메모리 절약).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'jjminu/polyglot58-chatbot'      # 내 모델 위에 이어서 학습
BASE_TOK   = 'EleutherAI/polyglot-ko-5.8b'    # 토크나이저 폴백

try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_TOK)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM', target_modules=['query_key_value'])  # polyglot=GPTNeoX
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. 토크나이즈 (프롬프트는 손실 제외, 답변만 학습)

In [ ]:
from datasets import Dataset

PROMPT_HEAD = ("### 질문: 아래 [문서] 내용만 근거로 질문에 한국어 한 문장으로만 답해줘. 문서에 답이 없으면 다른 말 없이 정확히 '제공된 문서에서 답을 찾을 수 없습니다'라고만 답해. 추측·부연 금지.\n\n[문서]\n{context}\n\n[질문] {question}\n\n### 답변:")

def encode(row):
    prompt = PROMPT_HEAD.format(context=row['context'], question=row['question'])
    full = prompt + ' ' + row['answer'] + tokenizer.eos_token
    p_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    f_ids = tokenizer(full, add_special_tokens=False, truncation=True, max_length=768)['input_ids']
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    labels = labels[:len(f_ids)]
    return {'input_ids': f_ids, 'labels': labels, 'attention_mask': [1] * len(f_ids)}

ds = Dataset.from_list([encode(r) for r in train_rows])
print(ds)

## 7. 학습 (LoRA)

In [ ]:
import torch
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments

@dataclass
class Collator:
    pad_id: int
    def __call__(self, feats):
        m = max(len(f['input_ids']) for f in feats)
        ii, ll, am = [], [], []
        for f in feats:
            pad = m - len(f['input_ids'])
            ii.append(f['input_ids'] + [self.pad_id] * pad)
            ll.append(f['labels'] + [-100] * pad)
            am.append(f['attention_mask'] + [0] * pad)
        return {'input_ids': torch.tensor(ii), 'labels': torch.tensor(ll),
                'attention_mask': torch.tensor(am)}

args = TrainingArguments(output_dir='out', per_device_train_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=12, learning_rate=2e-4,
    warmup_ratio=0.05, lr_scheduler_type='cosine', fp16=True, logging_steps=5,
    save_strategy='no', report_to='none', remove_unused_columns=False)

trainer = Trainer(model=model, args=args, train_dataset=ds,
    data_collator=Collator(tokenizer.pad_token_id))
trainer.train()

## 8. 학습 후 동작 테스트 (근거답변 + 거부)

In [ ]:
import torch
model.eval()

def gen(context, question, n=64):
    prompt = PROMPT_HEAD.format(context=context, question=question)
    ids = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, do_sample=False,
            repetition_penalty=1.2, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    t = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    for s in ['###', '\n']:
        i = t.find(s)
        if i != -1:
            t = t[:i]
    return t.strip()

adapterz = train_rows[0]['context']
refund = [r['context'] for r in train_rows if '350만' in r['context']][0]
print('근거답변:', gen(adapterz, '어댑터즈 구독료 얼마야?'))
print('근거답변:', gen(refund, '환불 언제까지 전액이야?'))
print('거부   :', gen(adapterz, '부트캠프 수강료 얼마야?'))
print('거부   :', gen(adapterz, '비트코인 사도 돼?'))

## 9. 병합 + HF Hub 업로드
LoRA 를 베이스에 병합해 완성 모델을 업로드한다. (fp16 재로드 — A100/L4 권장)

In [ ]:
import torch, gc, os
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

NEW_REPO = 'jjminu/polyglot58-rag'   # 원하는 이름으로 변경 가능

# 학습 직후라면 어댑터 저장 (이미 있으면 건너뜀 -> 재실행 안전)
if 'trainer' in globals() and not os.path.exists('lora_adapter/adapter_config.json'):
    trainer.model.save_pretrained('lora_adapter')
    (tokenizer if 'tokenizer' in globals() else AutoTokenizer.from_pretrained(BASE_TOK)).save_pretrained('lora_adapter')

assert os.path.exists('lora_adapter/adapter_config.json'), 'lora_adapter 없음 -> 학습 셀(7)부터 다시'

# 메모리 정리 후 fp16 로 베이스 재로드 -> 병합
for _v in ['model', 'trainer', 'base', 'merged']:
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, 'lora_adapter').merge_and_unload()
merged.push_to_hub(NEW_REPO)
try:
    AutoTokenizer.from_pretrained('lora_adapter').push_to_hub(NEW_REPO)
except Exception:
    AutoTokenizer.from_pretrained(BASE_TOK).push_to_hub(NEW_REPO)
print('업로드 완료 ->', NEW_REPO)

---
## 10. RAG 노트북에 연결
`otto_rag_my_model_colab.ipynb` 의 셀 5에서 모델 ID만 바꾸면 끝:
```python
MODEL_ID = 'jjminu/polyglot58-rag'
```
그 뒤 셀 6~11 을 다시 실행하면, **재학습된 내 모델**로 RAG + FastAPI + LangSmith 평가가 돌아갑니다.

결과가 아직 약하면: 학습셋(셀 4)에 예시를 더 추가하거나 `num_train_epochs` 를 늘려 재학습하세요.